# Bedrock Guardrail Manager

Idempotent guardrail lifecycle management — create, version, update, and publish.

| Cell | Purpose | Run |
|------|---------|-----|
| 1 | Config & imports | Always |
| 2 | Authenticate | Always |
| 3 | Create guardrail | Always (idempotent) |
| 4 | Publish version 1 | Always (idempotent) |
| 5 | Update DRAFT | Only when making changes |
| 6 | Publish new version | Only after Cell 5 |

## Cell 1 — Config & Imports

In [ ]:
import boto3
import json
from botocore.exceptions import ClientError

# =============================================================
# CONFIG — edit these values only
# =============================================================
GUARDRAIL_NAME     = "my-guardrail"                 # fixed name — never changes
BEDROCK_ACCOUNT_ID = "<BEDROCK_ACCOUNT_ID>"          # target AWS account
BEDROCK_ROLE_NAME  = "<BedrockCrossAccountRole>"     # cross-account role
REGION             = "us-east-1"

print(f"Config loaded — guardrail name: '{GUARDRAIL_NAME}'")

## Cell 2 — Authenticate (Assume Cross-Account Role)

In [ ]:
sts_client = boto3.client("sts")

assumed_role = sts_client.assume_role(
    RoleArn=f"arn:aws:iam::{BEDROCK_ACCOUNT_ID}:role/{BEDROCK_ROLE_NAME}",
    RoleSessionName="SageMakerGuardrailSession"
)
credentials = assumed_role["Credentials"]

bedrock_client = boto3.client(
    "bedrock",
    region_name=REGION,
    aws_access_key_id=credentials["AccessKeyId"],
    aws_secret_access_key=credentials["SecretAccessKey"],
    aws_session_token=credentials["SessionToken"]
)

print(f"✅ Authenticated — Bedrock client ready ({REGION})")

## Cell 3 — Create Guardrail *(idempotent)*

> Skips creation silently if a guardrail with `GUARDRAIL_NAME` already exists.  
> `guardrail_id` is always set after this cell — whether created or resolved.

In [ ]:
def get_guardrail_id_by_name(client, name):
    """
    list_guardrails (no identifier) returns one DRAFT entry per guardrail.
    Matches by fixed name + DRAFT version to resolve the guardrail ID.
    """
    paginator = client.get_paginator("list_guardrails")
    for page in paginator.paginate():
        for g in page.get("guardrails", []):
            if g["name"] == name and g["version"] == "DRAFT":
                return g["guardrailId"]
    return None


# --- Resolve or create ---
guardrail_id = get_guardrail_id_by_name(bedrock_client, GUARDRAIL_NAME)

if guardrail_id:
    print(f"⚠️  Guardrail '{GUARDRAIL_NAME}' already exists — skipping creation.")
    print(f"   ID: {guardrail_id}")
else:
    response = bedrock_client.create_guardrail(
        name=GUARDRAIL_NAME,
        description="Guardrail to filter harmful content",
        blockedInputMessaging="Sorry, I cannot process this request.",
        blockedOutputsMessaging="Sorry, I cannot return this response.",

        contentPolicyConfig={
            "filtersConfig": [
                {"type": "HATE",          "inputStrength": "HIGH",   "outputStrength": "HIGH"},
                {"type": "VIOLENCE",      "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "SEXUAL",        "inputStrength": "HIGH",   "outputStrength": "HIGH"},
                {"type": "INSULTS",       "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "MISCONDUCT",    "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "PROMPT_ATTACK", "inputStrength": "HIGH",   "outputStrength": "NONE"},
            ]
        },
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "Financial Advice",
                    "definition": "Providing specific financial or investment advice to users.",
                    "examples": [
                        "Should I buy Tesla stock?",
                        "What crypto should I invest in?"
                    ],
                    "type": "DENY"
                }
            ]
        },
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "PHONE", "action": "ANONYMIZE"},
                {"type": "SSN",   "action": "BLOCK"},
            ]
        },
        wordPolicyConfig={
            "wordsConfig": [
                {"text": "competitor_name"},
                {"text": "badword"}
            ],
            "managedWordListsConfig": [{"type": "PROFANITY"}]
        },
        tags=[
            {"key": "Environment", "value": "production"},
            {"key": "CreatedBy",   "value": "sagemaker"}
        ]
    )

    guardrail_id = response["guardrailId"]
    print(f"✅ Guardrail '{GUARDRAIL_NAME}' created:")
    print(f"   ID:      {guardrail_id}")
    print(f"   ARN:     {response['guardrailArn']}")
    print(f"   Version: {response['version']}")

## Cell 4 — Publish Version 1 *(idempotent)*

> Skips if version `1` already exists.  
> Uses `guardrail_id` + `GUARDRAIL_NAME` to double-verify the correct guardrail.

In [ ]:
def ensure_version_1(client, guardrail_id, guardrail_name):
    """
    list_guardrails with identifier returns all versions (DRAFT, 1, 2, ...).
    Creates version 1 only if it doesn't already exist.
    """
    response = client.list_guardrails(guardrailIdentifier=guardrail_id)
    existing_versions = [
        g["version"] for g in response.get("guardrails", [])
        if g["name"] == guardrail_name
        and g["id"] == guardrail_id
        and g["version"] != "DRAFT"
    ]

    print(f"   Existing numbered versions: {existing_versions}")

    if "1" in existing_versions:
        print(f"⚠️  Version 1 already exists for '{guardrail_name}' ({guardrail_id}) — skipping.")
    else:
        ver_response = client.create_guardrail_version(
            guardrailIdentifier=guardrail_id,
            description="First production version"
        )
        print(f"✅ Version {ver_response['version']} published for '{guardrail_name}' ({guardrail_id})")


ensure_version_1(bedrock_client, guardrail_id, GUARDRAIL_NAME)

## Cell 5 — Update Guardrail DRAFT

> **Only run when you want to make changes.**  
> Edits the DRAFT in-place. Numbered versions (1, 2, ...) are immutable.  
> After running this cell, run Cell 6 to publish the updated DRAFT as a new version.

In [ ]:
def update_guardrail(client, guardrail_id, guardrail_name):
    """
    Updates the DRAFT version of the guardrail.
    Modify the policy configs below to reflect your changes.
    name is required by the API even if it hasn't changed.
    """
    response = client.update_guardrail(
        guardrailIdentifier=guardrail_id,
        name=guardrail_name,                    # required even if unchanged
        description="Guardrail to filter harmful content — updated",
        blockedInputMessaging="Sorry, I cannot process this request.",
        blockedOutputsMessaging="Sorry, I cannot return this response.",

        contentPolicyConfig={
            "filtersConfig": [
                {"type": "HATE",          "inputStrength": "HIGH",   "outputStrength": "HIGH"},
                {"type": "VIOLENCE",      "inputStrength": "HIGH",   "outputStrength": "HIGH"},   # updated
                {"type": "SEXUAL",        "inputStrength": "HIGH",   "outputStrength": "HIGH"},
                {"type": "INSULTS",       "inputStrength": "HIGH",   "outputStrength": "HIGH"},   # updated
                {"type": "MISCONDUCT",    "inputStrength": "MEDIUM", "outputStrength": "MEDIUM"},
                {"type": "PROMPT_ATTACK", "inputStrength": "HIGH",   "outputStrength": "NONE"},
            ]
        },
        topicPolicyConfig={
            "topicsConfig": [
                {
                    "name": "Financial Advice",
                    "definition": "Providing specific financial or investment advice to users.",
                    "examples": [
                        "Should I buy Tesla stock?",
                        "What crypto should I invest in?"
                    ],
                    "type": "DENY"
                }
            ]
        },
        sensitiveInformationPolicyConfig={
            "piiEntitiesConfig": [
                {"type": "EMAIL", "action": "ANONYMIZE"},
                {"type": "PHONE", "action": "ANONYMIZE"},
                {"type": "SSN",   "action": "BLOCK"},
            ]
        },
        wordPolicyConfig={
            "wordsConfig": [
                {"text": "competitor_name"},
                {"text": "badword"}
            ],
            "managedWordListsConfig": [{"type": "PROFANITY"}]
        },
    )

    print(f"✅ Guardrail '{guardrail_name}' DRAFT updated:")
    print(f"   ID:      {response['guardrailId']}")
    print(f"   Version: {response['version']}")
    return response


update_guardrail(bedrock_client, guardrail_id, GUARDRAIL_NAME)

## Cell 6 — Publish New Version

> **Only run after Cell 5.**  
> Snapshots the updated DRAFT into the next numbered version (auto-increments).  
> Update `description` below to describe what changed in this version.

In [ ]:
def publish_new_version(client, guardrail_id, guardrail_name, description=None):
    """
    Snapshots current DRAFT into the next numbered version.
    Bedrock auto-increments: version 1 -> 2 -> 3, etc.
    """
    response = client.list_guardrails(guardrailIdentifier=guardrail_id)
    existing_versions = sorted([
        int(g["version"]) for g in response.get("guardrails", [])
        if g["name"] == guardrail_name
        and g["id"] == guardrail_id
        and g["version"] != "DRAFT"
    ])

    next_version = (existing_versions[-1] + 1) if existing_versions else 1
    print(f"   Current versions: {existing_versions} → publishing version {next_version}")

    ver_response = client.create_guardrail_version(
        guardrailIdentifier=guardrail_id,
        description=description or f"Version {next_version}"
    )

    print(f"✅ Version {ver_response['version']} published for '{guardrail_name}' ({guardrail_id})")
    return ver_response


publish_new_version(
    bedrock_client,
    guardrail_id,
    GUARDRAIL_NAME,
    description="Increased violence and insults strength to HIGH"   # describe what changed
)